In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

In [25]:
raw_path = Path("../data/raw")
csv_files = list(raw_path.glob("*.csv"))
print("CSV files found:")
for file in csv_files:
    print(file)

file_path = csv_files[0]
df = pd.read_csv(file_path)
print("\nDataset loaded successfully!")
print("Shape:", df.shape)

CSV files found:
..\data\raw\transaction_data.csv

Dataset loaded successfully!
Shape: (6362620, 11)


In [29]:
df.head()



,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [30]:
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Column names: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Data types:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [31]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print("Total missing values:", missing.sum())

duplicates = df.duplicated().sum()
print("\nDuplicate rows:", duplicates)

Missing values per column:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
Total missing values: 0

Duplicate rows: 0


In [32]:
fraud_count = df["isFraud"].sum()
total_transactions = len(df)
fraud_rate = fraud_count / total_transactions * 100

print("Fraud Statistics:")
print(f"Total transactions: {total_transactions:,}")
print(f"Fraud transactions: {fraud_count:,}")
print(f"Legitimate transactions: {total_transactions - fraud_count:,}")
print(f"Fraud rate: {fraud_rate:.4f}%")

Fraud Statistics:
Total transactions: 6,362,620
Fraud transactions: 8,213
Legitimate transactions: 6,354,407
Fraud rate: 0.1291%


In [33]:
fraud_by_type = df[df["isFraud"] == 1]["type"].value_counts()
print("Fraud transactions by type:")
print(fraud_by_type)
print("\n(Sanity check: should ONLY be TRANSFER and CASH_OUT)")

Fraud transactions by type:
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

(Sanity check: should ONLY be TRANSFER and CASH_OUT)


In [34]:
min_step = df["step"].min()
max_step = df["step"].max()
total_steps = max_step - min_step + 1

print("Minimum step:", min_step)
print("Maximum step:", max_step)
print("Total steps:", total_steps)

Minimum step: 1
Maximum step: 743
Total steps: 743


In [35]:
DELAY_DAYS = 7
DELAY_STEPS = DELAY_DAYS * 24

print(f"Delay window: {DELAY_DAYS} days ({DELAY_STEPS} steps)")
print("Total available steps:", total_steps)
print(f"Steps remaining after a {DELAY_DAYS}-day window:", total_steps - DELAY_STEPS)

Delay window: 7 days (168 steps)
Total available steps: 743
Steps remaining after a 7-day window: 575


In [36]:
df["day_bucket"] = (df["step"] - df["step"].min()) // 24
fraud_per_day = df[df["isFraud"] == 1].groupby("day_bucket").size()
print("Fraud cases per day:")
print(fraud_per_day)

df["week_bucket"] = (df["step"] - df["step"].min()) // (24 * 7)
fraud_per_week = df[df["isFraud"] == 1].groupby("week_bucket").size()
print("\nFraud cases per week (note: last week is partial, not a real drop):")
print(fraud_per_week)

Fraud cases per day:
day_bucket
0     271
1     309
2     310
3     262
4     252
5     228
6     272
7     278
8     255
9     282
10    262
11    298
12    242
13    246
14    250
15    252
16    320
17    268
18    256
19    236
20    272
21    256
22    216
23    280
24    240
25    272
26    280
27    248
28    260
29    268
30    272
dtype: int64

Fraud cases per week (note: last week is partial, not a real drop):
week_bucket
0    1904
1    1863
2    1854
3    1792
4     800
dtype: int64


In [37]:
fraud_df = df[df["isFraud"] == 1]

print("Fraud by origin account (repeat offenders):")
print(fraud_df["nameOrig"].value_counts().head(10))

print("\nFraud by destination account (repeat targets):")
print(fraud_df["nameDest"].value_counts().head(10))

print("\n--- OVERALL ACCOUNT REUSE (not just fraud subset) ---")
print(f"Unique nameOrig: {df['nameOrig'].nunique():,}")
print(f"nameOrig appearing more than once: {(df['nameOrig'].value_counts() > 1).sum():,}")
print(f"Unique nameDest: {df['nameDest'].nunique():,}")
print(f"nameDest appearing more than once: {(df['nameDest'].value_counts() > 1).sum():,}")

Fraud by origin account (repeat offenders):
nameOrig
C1305486145    1
C840083671     1
C1420196421    1
C2101527076    1
C137533655     1
C1118430673    1
C749981943     1
C1334405552    1
C467632528     1
C1364127192    1
Name: count, dtype: int64

Fraud by destination account (repeat targets):
nameDest
C410033330     2
C803116137     2
C904300960     2
C1013511446    2
C2020337583    2
C200064275     2
C185805228     2
C52390890      2
C935310781     2
C1827219533    2
Name: count, dtype: int64

--- OVERALL ACCOUNT REUSE (not just fraud subset) ---
Unique nameOrig: 6,353,307
nameOrig appearing more than once: 9,298
Unique nameDest: 2,722,362
nameDest appearing more than once: 459,658


In [38]:
print("isFlaggedFraud value counts:")
print(df["isFlaggedFraud"].value_counts())
print("\nFraud vs isFlaggedFraud:")
print(pd.crosstab(df["isFraud"], df["isFlaggedFraud"], margins=True))

isFlaggedFraud value counts:
isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

Fraud vs isFlaggedFraud:
isFlaggedFraud        0   1      All
isFraud                             
0               6354407   0  6354407
1                  8197  16     8213
All             6362604  16  6362620


FEATURE ENGINEERING............

In [39]:
df["timestamp"] = pd.to_datetime(df["step"], unit="h", origin="2023-01-01")
print("Timestamp column created.")
print(df[["step", "timestamp"]].head())

Timestamp column created.
   step           timestamp
0     1 2023-01-01 01:00:00
1     1 2023-01-01 01:00:00
2     1 2023-01-01 01:00:00
3     1 2023-01-01 01:00:00
4     1 2023-01-01 01:00:00


In [40]:
df = df.reset_index().rename(columns={"index": "original_order"})
df = df.sort_values(["nameDest", "timestamp"])
print("Sorted for rolling computation.")
print(f"Rows: {len(df):,}")

Sorted for rolling computation.
Rows: 6,362,620


In [41]:
grouped = df.set_index("timestamp").groupby("nameDest")["amount"]

df["destination_transactions_last_24h"] = (
    grouped.rolling("24h", closed="left").count().reset_index(drop=True)
)
df["destination_transactions_last_7d"] = (
    grouped.rolling("168h", closed="left").count().reset_index(drop=True)
)

print("Destination rolling counts computed.")
print(df[["nameDest", "timestamp", "destination_transactions_last_24h",
          "destination_transactions_last_7d"]].head(10))

Destination rolling counts computed.
            nameDest           timestamp  destination_transactions_last_24h  \
4987517  C1000004082 2023-01-15 16:00:00                                NaN   
5032095  C1000004082 2023-01-15 18:00:00                                NaN   
5219127  C1000004082 2023-01-16 10:00:00                                NaN   
5331822  C1000004082 2023-01-16 14:00:00                                NaN   
5462768  C1000004082 2023-01-16 19:00:00                                NaN   
5659867  C1000004082 2023-01-17 12:00:00                                NaN   
1148643  C1000004940 2023-01-06 11:00:00                                NaN   
1165745  C1000004940 2023-01-06 12:00:00                                NaN   
1762000  C1000004940 2023-01-07 17:00:00                                3.0   
2020611  C1000004940 2023-01-08 12:00:00                                1.0   

         destination_transactions_last_7d  
4987517                               NaN  
50320

In [42]:
df["destination_avg_previous_amount"] = (
    df.groupby("nameDest")["amount"]
      .apply(lambda x: x.shift(1).expanding().mean())
      .reset_index(drop=True)
)

df["destination_is_first_transaction"] = df["destination_avg_previous_amount"].isna().astype(int)
df["destination_avg_previous_amount"] = df["destination_avg_previous_amount"].fillna(0)

print("Destination average previous amount computed.")
print(f"First-time destinations: {df['destination_is_first_transaction'].sum():,}")

Destination average previous amount computed.
First-time destinations: 2,722,362


In [43]:
df["destination_amount_deviation"] = df["amount"] - df["destination_avg_previous_amount"]
print("Destination amount deviation computed.")
print(df["destination_amount_deviation"].describe())

Destination amount deviation computed.
count    6.362620e+06
mean     3.431261e+04
std      6.884530e+05
min     -3.967488e+07
25%     -1.354149e+05
50%      9.038280e+03
75%      1.219456e+05
max      9.233041e+07
Name: destination_amount_deviation, dtype: float64


In [44]:
df = df.sort_values("original_order").drop(columns="original_order").reset_index(drop=True)
print("Original row order restored.")

Original row order restored.


In [45]:
df["origin_balance_error"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["destination_balance_error"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

print("Balance consistency features created.")
print("\nOrigin balance error:")
print(df["origin_balance_error"].describe())
print("\nDestination balance error:")
print(df["destination_balance_error"].describe())

Balance consistency features created.

Origin balance error:
count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: origin_balance_error, dtype: float64

Destination balance error:
count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
Name: destination_balance_error, dtype: float64


In [46]:
step_stats = (
    df.groupby("step")["amount"]
      .agg(total_transactions="count", total_transaction_amount="sum",
           avg_transaction_amount="mean")
      .reset_index()
)
df = df.merge(step_stats, on="step", how="left")
print("Step-level features created.")
print(df[["step", "total_transactions", "total_transaction_amount",
          "avg_transaction_amount"]].head())

Step-level features created.
   step  total_transactions  total_transaction_amount  avg_transaction_amount
0     1                2708              2.854292e+08           105402.208696
1     1                2708              2.854292e+08           105402.208696
2     1                2708              2.854292e+08           105402.208696
3     1                2708              2.854292e+08           105402.208696
4     1                2708              2.854292e+08           105402.208696


In [47]:
print("Correlation with isFraud:")
print(df[["origin_balance_error", "destination_balance_error",
          "destination_amount_deviation"]].corrwith(df["isFraud"]))

df["destination_balance_is_zero"] = (
    (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0)
).astype(int)

print("\nCrosstab: destination_balance_is_zero vs isFraud (row %)")
print(pd.crosstab(df["destination_balance_is_zero"], df["isFraud"], normalize="index"))

Correlation with isFraud:
origin_balance_error            0.011283
destination_balance_error       0.055120
destination_amount_deviation    0.070493
dtype: float64

Crosstab: destination_balance_is_zero vs isFraud (row %)
isFraud                             0         1
destination_balance_is_zero                    
0                            0.998977  0.001023
1                            0.998241  0.001759


In [48]:
print("FINAL FEATURE PLAN")
print("\nTransaction features: amount, type")
print("\nDestination history features:")
for f in ["destination_transactions_last_24h", "destination_transactions_last_7d",
          "destination_avg_previous_amount", "destination_amount_deviation",
          "destination_is_first_transaction"]:
    print("-", f)
print("\nBalance consistency features:")
for f in ["origin_balance_error", "destination_balance_error", "destination_balance_is_zero"]:
    print("-", f)
print("\nSystem-level features:")
for f in ["total_transactions", "total_transaction_amount", "avg_transaction_amount"]:
    print("-", f)
print("\nNote: account age and device/location flags dropped — PaySim's schema")
print("and near-total origin-account non-reuse don't support them.")

FINAL FEATURE PLAN

Transaction features: amount, type

Destination history features:
- destination_transactions_last_24h
- destination_transactions_last_7d
- destination_avg_previous_amount
- destination_amount_deviation
- destination_is_first_transaction

Balance consistency features:
- origin_balance_error
- destination_balance_error
- destination_balance_is_zero

System-level features:
- total_transactions
- total_transaction_amount
- avg_transaction_amount

Note: account age and device/location flags dropped — PaySim's schema
and near-total origin-account non-reuse don't support them.
